In [ ]:
#| default_exp preprocessing.rot_image

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from pathlib import Path
import tensorflow as tf
from fastcore.all import *
import numpy as np
import os
import sys
import yaml
import cv2
from fastcore.imports import *
from PIL import Image

In [ ]:
path_ =Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1408/missing_1408_unzip/main_im2_cropped_masks/threshold52_new/failed/additional/images')
path_.ls()

In [ ]:
from cv_tools.core import *

In [ ]:
im_fn = path_.ls()[0]

In [ ]:
img_ = read_img(im_fn)

In [ ]:
show_(img_)

In [ ]:
nm = Path(im_fn).name
fn_ = fr'M:\Fail_investigation\Missing_lead\ETPD_WAR_1_02_missing_20240813\ETPD_WAR_1_02_missing_20240813_unzip\main_im2_cropped_masks\threshold52\failed\missing\missing_pin_sn_images\{nm}' 

In [ ]:
fn_=Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/ETPD_WAR_1_02_missing_20240813_unzip/main_im2_cropped_masks/threshold52/failed/missing/missing_pin_sn_images/tst.png')

In [ ]:
dst_path = Path('/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/ETPD_WAR_1_02_missing_20240813_unzip/main_im2_cropped_masks/threshold52_new/failed/missing/missing_pin_sn_images')

In [ ]:
dst_path.ls()

In [ ]:
show_(img_[340:440, 1070:1168])

In [ ]:
cv2.imwrite(
    f'{dst_path}/tst.png',
    img_[340:440, 1070:1168]
    )

In [ ]:
import shutil
import os

In [ ]:
path =Path(r'M:\Fail_investigation\Missing_lead\ETPD_WAR_1_02_missing_20240813\ETPD_WAR_1_02_missing_20240813_unzip\main_im2_cropped_masks\threshold52\failed\missing\missing_pin_sn_images')
path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/ETPD_WAR_1_02_missing_20240813_unzip/main_im2_cropped_masks/threshold52_new/failed/missing/missing_pin_sn_images')

In [ ]:
path.ls()

In [ ]:

for idx, i in enumerate(path.ls()):
    name_ = i.name
    pat = 'In_125_.*.png'
    pat = re.compile(pat)
    pa_ = re.findall(pat, name_)
    if pa_:
        nm = pa_[0]
        new_name = f'rec_{idx}_{nm}'
        Path(i).rename(Path(path,new_name))

In [ ]:
ffn = Path(fn_)

In [ ]:
show_(img_)

In [ ]:
os.getenv('PATH')
custom_lib_path = Path("/home/ai_warstein/homes/goni/custom_libs")
sys.path.append(str(custom_lib_path))
from dotenv import load_dotenv
load_dotenv(dotenv_path="/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/.env")

In [ ]:
CV_TOOLS = os.getenv("CV_TOOLS")
P_EPD= os.getenv("PRIVATE_EASY_PIN_DETECTION")
sys.path.append(CV_TOOLS)

In [ ]:
from cv_tools.core import *

In [ ]:
im_path = Path('/home/ai_easypid.work/Reference_images')
ref_im2b = Path(im_path, 'ref_image_2B.png')

In [ ]:
img = read_img(ref_im2b)

In [ ]:
n_img = rotate_image(img, 7)
show_(n_img)

In [ ]:
sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(img, None)
kp2, des2 = sift.detectAndCompute(n_img, None)

In [ ]:
# FlANN parameters
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des1, des2, k=2)
good_matches = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good_matches.append(m)

In [ ]:
calculate_rotation(kp1, kp2, good_matches)

In [ ]:
def detect_features(
    img1: np.ndarray,
    img2: np.ndarray,
    method:str, # whether to use SIFT or ORB algorithm
    ) -> tuple[cv2.KeyPoint, cv2.KeyPoint, cv2.DMatch]:
    ' find best feature'

    if method == 'SIFT':
        detector = cv2.SIFT_create()
        kp1, des1 = detector.detectAndCompute(img1, None)
        kp2, des2 = detector.detectAndCompute(img2, None)

        FLANN_INDEX_KDTREE = 1
        index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
        search_params = dict(checks=50)
        flann = cv2.FlannBasedMatcher(
            index_params,
            search_params

            )
        matches = flann.knnMatch(des1, des2, k=2)
        good_matches =[] 
        for m,n in matches:
            if m.distance < 0.7 * n.distance:
                good_matches.append(m)
        return kp1, kp2, good_matches
    elif method == 'ORB':
        detector = cv2.ORB_create()
        kp1, des1 = detector.detectAndCompute(img1, None)
        kp2, des2 = detector.detectAndCompute(img2, None)
        bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        matches = bf.match(des1, des2)
        matches = sorted(matches, key = lambda x:x.distance)
        good_matches = matches[:10]
        return kp1, kp2, good_matches

In [ ]:
def calculate_rotation(
    kp1:cv2.KeyPoint, # keypoint of image 1 after feature detection
    kp2:cv2.KeyPoint, # keypoint of image 2 after feature detection
    good_matches : cv2.DMatch # good matches between the two images
    )-> float: # Angle of the rotation between the two images
    src_pts = np.float32(
        [kp1[m.queryIdx].pt for m in good_matches]
        ).reshape(-1, 1, 2)
    dst_pts = np.float32(
        [kp2[m.queryIdx].pt for m in good_matches]
    ).reshape(-1, 1, 2)
    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    angle = np.arctan2(M[1, 0], M[0, 0]) * 180 / np.pi

     
    # Normalize the angle to be between -180 and 180
    angle = (angle + 180) % 360 - 180

    if angle < -90:
        angle += 360
    return angle

In [ ]:
def calculate_rotationv02(
    kp1:cv2.KeyPoint, # keypoint of image 1 after feature detection
    kp2:cv2.KeyPoint, # keypoint of image 2 after feature detection
    good_matches : cv2.DMatch # good matches between the two images
    )-> float: # Angle of the rotation between the two images
    src_pts = np.float32(
        [kp1[m.queryIdx].pt for m in good_matches]
        ).reshape(-1,  2)
    dst_pts = np.float32(
        [kp2[m.queryIdx].pt for m in good_matches]
    ).reshape(-1,  2)


    # Calculate the center of mass of the points
    src_center = np.mean(src_pts, axis=0)
    dst_center = np.mean(dst_pts, axis=0)

    # Substruct the center to align the points
    src_centered = src_pts - src_center
    dst_centered = dst_pts - dst_center

    # Calculate the rotation angle
    angles = np.arctan2(
        dst_centered[:, 1], 
        dst_centered[:, 0]
        ) - np.arctan2(src_centered[:, 1], src_centered[:, 0])
    
    # Normalize angles to [-pi, pi]
    angles = (angles + np.pi) % (2 * np.pi) - np.pi

    # Calculate the median angle
    rotation_angle = np.median(angles) * 180 / np.pi

    return rotation_angle




In [ ]:
rows, cols = img.shape[:2]
center = (cols/2, rows/2)
rot_mat = cv2.getRotationMatrix2D(center, 0, 1.0)

In [ ]:
rotated = cv2.warpAffine(img, rot_mat, (cols, rows))

In [ ]:
rot_mat

In [ ]:
rotated.shape

In [ ]:
img.shape

In [ ]:
cv2.findTransformECC?

In [ ]:
cv2.findTransformECC(img, rotated, rot_mat, cv2.MOTION_EUCLIDEAN)

In [ ]:
def calculate_rotationv03(
    image1:np.ndarray,
    image2:np.ndarray,
    )-> float: # Angle of the rotation between the two images

    rows, cols = image1.shape[:2]
    center = (cols/2, rows/2)

    rot_mat = cv2.getRotationMatrix2D(center, 0, 1.0)
    rotated = cv2.warpAffine(image2, rot_mat, (cols, rows))

    h = cv2.findTransformECC(
        image1, 
        rotated, 
        rot_mat, 
        cv2.MOTION_EUCLIDEAN, 
        )[0]
    angle = np.arctan2(h[1, 0], h[0, 0]) * 180 / np.pi
    return angle
ang_ = calculate_rotationv03(img, n_img)

In [ ]:
show_(img)

In [ ]:
show_(n_img)

In [ ]:
180-179

In [ ]:
key1, key2, good_matches_f = detect_features(img, n_img, 'SIFT')
rot_angle_ = calculate_rotationv02(key1, key2, good_matches_f)
print(f' Rotation angle is {rot_angle_} degrees')
rotated_img = rotate_image(n_img, (rot_angle_))
show_(rotated_img)

In [ ]:
key1, key2, good_matches_f = detect_features(img, n_img, 'ORB')
rot_angle_ = calculate_rotation(key1, key2, good_matches_f)
print(f' Rotation angle is {rot_angle_} degrees')
rotated_img = rotate_image(n_img, (rot_angle_))
show_(rotated_img)

In [ ]:
show_(img)

In [ ]:
key1, key2, good_matches_f = detect_features(img, n_img, 'SIFT')
rot_angle_ = calculate_rotation(key1, key2, good_matches_f)
print(f' Rotation angle is {rot_angle_} degrees')
rotated_img = rotate_image(n_img, rot_angle_)
show_(rotated_img)

In [ ]:
show_(img)

In [ ]:
ang_ = calculate_rotation(kp1, kp2, good_matches)

In [ ]:
rot_img = rotate_image(n_img,ang_ )

In [ ]:
show_(rot_img)

In [ ]:
orb = cv2.ORB_create(50)
kp1, des1 = orb.detectAndCompute(img, None)
kp2, des2 = orb.detectAndCompute(n_img, None)
matcher = cv2.DescriptorMatcher_create(cv2.DESCRIPTOR_MATCHER_BRUTEFORCE_HAMMING)
matches = matcher.match(des1, des2, None)

im3 = cv2.drawMatches(
    img, 
    kp1, 
    n_img, 
    kp2, 
    matches[:10], 
    None, 
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

In [ ]:
show_(im3)

In [ ]:
pt1 = np.zeros(
    (len(matches), 2), dtype=np.float32) 
pt2 = np.zeros(
    (len(matches), 2), dtype=np.float32
    )

In [ ]:
for i, matches in enumerate(matches):
    pt1[i, :] = kp1[matches.queryIdx].pt
    pt2[i, :] = kp2[matches.trainIdx].pt

In [ ]:
h, mask = cv2.findHomography(pt1, pt2, cv2.RANSAC)

In [ ]:
new_img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)

In [ ]:
show_(img)

In [ ]:
show_(new_img)

In [ ]:
import cv2
import numpy as np

def detect_circle(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)
    circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                               param1=50, param2=30, minRadius=10, maxRadius=100)
    
    if circles is not None:
        circles = np.round(circles[0, :]).astype("int")
        x, y, r = circles[0]
        return (x, y)
    return None

def calculate_angle(point1, point2):
    dx = point2[0] - point1[0]
    dy = point2[1] - point1[1]
    return np.degrees(np.arctan2(dy, dx))

def rotate_image(image, angle):
    center = tuple(np.array(image.shape[1::-1]) / 2)
    rot_mat = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(image, rot_mat, image.shape[1::-1], flags=cv2.INTER_LINEAR)

# Load reference and test images
reference_image = cv2.imread('reference_image.jpg')
test_image = cv2.imread('test_image.jpg')

# Detect circles in both images
ref_center = detect_circle(reference_image)
test_center = detect_circle(test_image)

if ref_center is None or test_center is None:
    print("Could not detect circles in one or both images.")
else:
    # Calculate the angle difference
    ref_angle = calculate_angle(ref_center, (ref_center[0], 0))
    test_angle = calculate_angle(test_center, (test_center[0], 0))
    rotation_angle = ref_angle - test_angle

    # Rotate the test image
    rotated_test_image = rotate_image(test_image, rotation_angle)

    # Display results
    cv2.imshow('Reference Image', reference_image)
    cv2.imshow('Original Test Image', test_image)
    cv2.imshow('Rotated Test Image', rotated_test_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    # Save the rotated test image
    cv2.imwrite('rotated_test_image.jpg', rotated_test_image)